# Aiqon Assignment: Train Models

This notebook trains the ETA regression models and the Delay Risk classification models for the shipment dataset. It executes the same code as `scripts/train_models.py` but in an interactive format.

In [5]:
import os

# Unzip the uploaded project
!unzip -o -q /content/test2.zip-d /content/shipment_project

# Change the current working directory to the project folder
os.chdir('/content/shipment_project')

# Install the required libraries for the project
!pip install -r requirements.txt -q

print("Project unzipped and ready! Current directory is:", os.getcwd())


unzip:  cannot find or open /content/test2.zip-d, /content/test2.zip-d.zip or /content/test2.zip-d.ZIP.


FileNotFoundError: [Errno 2] No such file or directory: '/content/shipment_project'

In [14]:
import os
import sys
import zipfile

zip_path = '/content/test2.zip'
target_dir = '/content/shipment_project'

if not os.path.exists(zip_path):
    print(f"Error: The file {zip_path} does not exist at all in the /content directory.")
else:
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            print("Zip file integrity checked. Extracting...")
            zip_ref.extractall(target_dir)
            print(f"Successfully unzipped to {target_dir}!")

            # Let's locate the true directory that contains 'src'
            project_root = None
            for root, dirs, files in os.walk(target_dir):
                if 'src' in dirs:
                    # Ignore virtual environments and git folders
                    if '.venv' not in root and '.git' not in root:
                        project_root = root
                        break

            if project_root:
                print(f"Found true project root containing 'src' at: {project_root}")
                os.chdir(project_root)
                if os.getcwd() not in sys.path:
                    sys.path.insert(0, os.getcwd())
                print("Current working directory updated to:", os.getcwd())

                # Look for requirements.txt in this true project root
                req_path = os.path.join(project_root, 'requirements.txt')
                if os.path.exists(req_path):
                    print("Installing dependencies from requirements.txt...")
                    !pip install -r requirements.txt -q
                    print("Dependencies installed.")
                else:
                    print("Warning: No requirements.txt found in the located project root.")

                # Safely perform imports now
                import numpy as np
                import pandas as pd
                from sklearn.model_selection import train_test_split
                import json

                global CONFIG, logger, load_raw_data, preprocess, save_processed
                global build_feature_matrix, NUMERIC_FEATURES, CATEGORICAL_FEATURES
                global train_eta_models, save_model, train_classifiers, save_classifier

                from src import CONFIG, logger
                from src.data_loader import load_raw_data
                from src.preprocessing import preprocess, save_processed
                from src.feature_engineering import (
                    build_feature_matrix, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
                )
                from src.eta_model import train_eta_models, save_model
                from src.delay_classifier import train_classifiers, save_classifier

                print("All modules and 'CONFIG' imported successfully!")
            else:
                print(f"[ERROR] Could not find any directory containing 'src' inside {target_dir}.")

    except zipfile.BadZipFile:
        print("\n[CRITICAL ERROR] The file '/content/test2.zip' is corrupted.")

Zip file integrity checked. Extracting...
Successfully unzipped to /content/shipment_project!
Found true project root containing 'src' at: /content/shipment_project/test2
Current working directory updated to: /content/shipment_project/test2
Installing dependencies from requirements.txt...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 61.2 MB/s eta 0:00:00
Dependencies installed.
All modules and 'CONFIG' imported successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Phase 1: Load and Preprocess Data

In [11]:
seed = CONFIG["project"]["random_seed"]
test_size = CONFIG["data"]["test_size"]

print("=== Phase 1: Load & Preprocess ===")
raw = load_raw_data('../data/raw/logistics-shipment-delay-prediction.xlsx')
df = preprocess(raw)
save_processed(df, '../data/processed/shipments_cleaned.csv')

# Filter to records with valid outcomes
df_train_pool = df[df["has_outcome"] == True].copy()
print(f"Records with valid outcome: {len(df_train_pool)}")

# Build feature matrix
X_all, num_cols, cat_cols = build_feature_matrix(df_train_pool)
y_reg = df_train_pool["delay_target_minutes"].values
y_clf = df_train_pool["is_delayed"].astype(int).values

# Train/test split
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = (
    train_test_split(
        X_all, y_reg, y_clf,
        test_size=test_size,
        random_state=seed,
        stratify=y_clf,
    )
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

NameError: name 'CONFIG' is not defined

In [ ]:
seed = CONFIG["project"]["random_seed"]
test_size = CONFIG["data"]["test_size"]

print("=== Phase 1: Load & Preprocess ===")
raw = load_raw_data('../data/raw/logistics-shipment-delay-prediction.xlsx')
df = preprocess(raw)
save_processed(df, '../data/processed/shipments_cleaned.csv')

# Filter to records with valid outcomes
df_train_pool = df[df["has_outcome"] == True].copy()
print(f"Records with valid outcome: {len(df_train_pool)}")

# Build feature matrix
X_all, num_cols, cat_cols = build_feature_matrix(df_train_pool)
y_reg = df_train_pool["delay_target_minutes"].values
y_clf = df_train_pool["is_delayed"].astype(int).values

# Train/test split
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = (
    train_test_split(
        X_all, y_reg, y_clf,
        test_size=test_size,
        random_state=seed,
        stratify=y_clf,
    )
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

NameError: name 'CONFIG' is not defined

### Phase 2: Train ETA Regression Models

In [10]:
print("=== Phase 2: ETA Regression ===")
eta_result = train_eta_models(
    X_train, y_reg_train,
    preprocessor=None,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_name = eta_result["best_name"]
best_model = eta_result["models"][best_name]
feature_cols = num_cols + cat_cols

save_model(
    best_model, best_name,
    eta_result["cv_results"][best_name],
    feature_cols,
    model_dir='../models'
)

print(f"\nETA CV results: {eta_result['cv_results']}")
print(f"Best ETA model: {best_name}")

=== Phase 2: ETA Regression ===


NameError: name 'train_eta_models' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Phase 1: Load and Preprocess Data

In [15]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import json

# Ensure '/content/shipment_project/test2' is in sys.path
project_path = '/content/shipment_project/test2'
if os.path.exists(project_path):
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    os.chdir(project_path)

# Now import from the local 'src' directory securely
from src import CONFIG, logger
from src.data_loader import load_raw_data
from src.preprocessing import preprocess, save_processed
from src.feature_engineering import (
    build_feature_matrix, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
)

seed = CONFIG["project"]["random_seed"]
test_size = CONFIG["data"]["test_size"]

print("=== Phase 1: Load & Preprocess ===")
# Load using path relative to current project directory
raw = load_raw_data('data/raw/logistics-shipment-delay-prediction.xlsx')
df = preprocess(raw)
save_processed(df, 'data/processed/shipments_cleaned.csv')

# Filter to records with valid outcomes
df_train_pool = df[df["has_outcome"] == True].copy()
print(f"Records with valid outcome: {len(df_train_pool)}")

# Build feature matrix
X_all, num_cols, cat_cols = build_feature_matrix(df_train_pool)
y_reg = df_train_pool["delay_target_minutes"].values
y_clf = df_train_pool["is_delayed"].astype(int).values

# Train/test split
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = (
    train_test_split(
        X_all, y_reg, y_clf,
        test_size=test_size,
        random_state=seed,
        stratify=y_clf,
    )
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

=== Phase 1: Load & Preprocess ===


Records with valid outcome: 145
Train size: 116 | Test size: 29


### Phase 2: Train ETA Regression Models

In [16]:
from src.eta_model import train_eta_models, save_model

print("=== Phase 2: ETA Regression ===")
eta_result = train_eta_models(
    X_train, y_reg_train,
    preprocessor=None,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_name = eta_result["best_name"]
best_model = eta_result["models"][best_name]
feature_cols = num_cols + cat_cols

save_model(
    best_model, best_name,
    eta_result["cv_results"][best_name],
    feature_cols,
    model_dir='models'
)

print(f"\nETA CV results: {eta_result['cv_results']}")
print(f"Best ETA model: {best_name}")

=== Phase 2: ETA Regression ===

ETA CV results: {'baseline_mean': {'mae_mean': 34.31258765778402, 'mae_std': 5.492590996902107}, 'ridge': {'mae_mean': 20.09126536448226, 'mae_std': 4.40614722503838, 'best_alpha': 1.0}, 'random_forest': {'mae_mean': 21.354892815027593, 'mae_std': 3.328123225450447, 'best_params': {'max_depth': 5, 'min_samples_leaf': 5, 'n_estimators': 50}}}
Best ETA model: ridge


### Phase 3: Train Delay Classifiers

In [18]:
from src.delay_classifier import train_classifiers, save_classifier

print("=== Phase 3: Delay Classification ===")
clf_result = train_classifiers(
    X_train, y_clf_train,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_clf_name = clf_result["best_name"]
best_clf = clf_result["models"][best_clf_name]
feature_cols = num_cols + cat_cols

save_classifier(
    best_clf, best_clf_name,
    clf_result["cv_results"][best_clf_name],
    feature_cols,
    model_dir='models'
)

print(f"\nClassifier CV results: {clf_result['cv_results']}")
print(f"Best classifier: {best_clf_name}")

=== Phase 3: Delay Classification ===


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sk


Classifier CV results: {'logistic_regression': {'f1_mean': 0.9139524314497803, 'f1_std': 0.03450074568170093, 'roc_auc_mean': nan, 'roc_auc_std': nan}, 'random_forest': {'f1_mean': 0.9684719535783366, 'f1_std': 0.018335496055944374, 'roc_auc_mean': nan, 'roc_auc_std': nan, 'best_params': {'max_depth': 5, 'min_samples_leaf': 3, 'n_estimators': 100}}}
Best classifier: random_forest


### Phase 4: Save Split Info for Evaluation

In [20]:
split_info = {
    "train_indices": X_train.index.tolist(),
    "test_indices": X_test.index.tolist(),
}
split_path = os.path.abspath(os.path.join(CONFIG["paths"]["models_dir"], "split_info.json"))
os.makedirs(os.path.dirname(split_path), exist_ok=True)
with open(split_path, "w") as f:
    json.dump(split_info, f)

print("=== Training Complete ===")

=== Training Complete ===
